## Convertir un caso clínico a FHIR

Ana Torres, 35 años, cédula 1020456789, Bogotá.
Consulta médica general el 7 de marzo de 2026.
Diagnóstico: Infección aguda de vías respiratorias superiores (J06.9)
Médico tratante: Dr. Carlos Méndez, registro médico 12345
IPS: Clínica Santa María, NIT 900123456-1

In [4]:

# CELDA 1 — El Paciente
# En FHIR todo recurso es un diccionario JSON
# La primera clave siempre es "resourceType" 
# Es lo que le dice a cualquier sistema del mundo: 
# "esto es un paciente, no un diagnóstico, no un medicamento"

import json

paciente = {
    "resourceType": "Patient", #lo que viene adentro de este diccionario es un paciente. Si fuera un diagnostico seria "Condition". Si fuera una consulta seria "Encaunter"
    "id": "pac-001", #Es el número de identificación interno del recurso dentro de tu sistema.Es como el consecutivo que le da tu software a este registro.
    "identifier": [ #Ceduña real de Ana. Va en [] porque es una lista. Una persona puede tener varios identificadores.
        {
            "system": "https://www.registraduria.gov.co", #le dice de dónde viene ese número. En Colombia usamos la URL de la Registraduría para decir que es una cédula colombiana. Es como decir "este número lo emitió este ente."
            "value": "1020456786" #numero en si de la cedula
        }
    ],
    "name": [
        {
            "family": "Corredor", # el apeillido. En FHIR se llama "family" no "lastName"
            "given": ["Francisco"] #los nombres. También es lista porque una persona puede tener dos nombres: Ana María.
        }
    ],
    "gender": "male", #FHIR acepta exactamente cuatro valores: "male", "female", "other", "unknown"
    "birthDate": "1990-12-08", #El formato siempre es AAAA-MM-DD
    "address": [
        {
            "city": "Cartagena",
            "country": "CO" # país en código ISO de dos letras. Colombia es "CO", no "Colombia".
        }
    ]
}

print(json.dumps(paciente, indent=2, ensure_ascii=False)) # Esta línea imprime el diccionario bonito y legible. Tiene tres partes:
#json.dumps(paciente) — convierte el diccionario de Python a texto JSON puro.
#indent=2 — le dice que use 2 espacios de sangría para que se vea organizado, no todo en una línea.
#ensure_ascii=False — le dice que respete tildes y caracteres especiales del español. Sin esto, "Bogotá" saldría como "Bogot\u00e1"
#L RESULTADO ES UN RECURSO FHIR. Si agarro ese texto y lo envío a un servidor FHIR, lo entiende. Python no transformó nada — solo lo mostró bonito.

{
  "resourceType": "Patient",
  "id": "pac-001",
  "identifier": [
    {
      "system": "https://www.registraduria.gov.co",
      "value": "1020456786"
    }
  ],
  "name": [
    {
      "family": "Corredor",
      "given": [
        "Francisco"
      ]
    }
  ],
  "gender": "male",
  "birthDate": "1990-12-08",
  "address": [
    {
      "city": "Cartagena",
      "country": "CO"
    }
  ]
}


## Encounter

Ana Torres ya está registrada en el sistema. Ahora entra al consultorio. El médico abre el sistema y registra la atención — fecha, hora, tipo de consulta, en qué IPS.
Eso es el Encounter. Es el registro de que Ana y el médico se encontraron en un momento específico.
Sin el Encounter, el diagnóstico y el medicamento quedan flotando en el aire — no se sabe a qué consulta pertenecen.

In [6]:
# CELDA 2 — La Consulta (Encounter)
# Registra el momento en que Ana fue atendida

consulta = {
    "resourceType": "Encounter",
    "id": "enc-001",
    "status": "finished", #Le dice al sistema en qué estado está esta consulta. FHIR acepta varios valores:"planned"La cita está agendada, no ha ocurrido"in-progress"El paciente está siendo atendido ahora mismo"finished"La consulta terminó"cancelled"La cita fue cancelada
    "class": { #"Define el tipo de encuentro clínico. ¿Fue ambulatorio, urgencias, hospitalización?
        "system": "http://terminology.hl7.org/CodeSystem/v3-ActCode", #de dónde viene el código. Aquí usa el catálogo internacional de HL7.
        "code": "AMB", #ambulatorio. Los más comunes en Colombia son: "AMB"Ambulatorio — consulta externa"EMER"Urgencias"IMP"Hospitalización
        "display": "ambulatorio" #el texto legible para humanos. El código es para máquinas, el display es para personas.
    },
    "subject": {
        "reference": "Patient/pac-001" #Está diciendo: *"esta consulta le pertenece al paciente cuyo id es pac-001.El `Encounter` no repite todos los datos de Ana — simplemente **apunta** al recurso `Patient` donde ya están. Esto es lo que hace FHIR poderoso: los recursos se conectan entre sí como una red.
    },
    "period": { #El tiempo que duró la consulta
        "end": "2026-03-07T09:30:00"
    }
}

print(json.dumps(consulta, indent=2, ensure_ascii=False))

{
  "resourceType": "Encounter",
  "id": "enc-001",
  "status": "finished",
  "class": {
    "system": "http://terminology.hl7.org/CodeSystem/v3-ActCode",
    "code": "AMB",
    "display": "ambulatorio"
  },
  "subject": {
    "reference": "Patient/pac-001"
  },
  "period": {
    "end": "2026-03-07T09:30:00"
  }
}


## Condition (diagnostico)

Después de examinar a Ana, tú como médica escribes el diagnóstico en la historia clínica. Pones el código CIE-10, el nombre, y lo asocias a esa consulta específica.
Eso es el Condition. Es el diagnóstico de Ana, asociado a la consulta de hoy.

In [10]:
# CELDA 3 — El Diagnóstico (Condition)
# El diagnóstico de Ana con su código CIE-10

diagnostico = {
    "resourceType": "Condition",
    "id": "con-001",
    "clinicalStatus": { #Le dice al sistema en qué estado está el diagnóstico clínicamente. ¿Sigue activo? ¿Ya se resolvió? ¿Es recurrente?. Los valores más comunes:"active"El diagnóstico está activo ahora mismo"resolved"Ya se resolvió — el paciente se curó"recurrence"Es una recaída de algo que ya había tenido"inactive"Ya no está activo pero no se resolvió completamente
        "coding": [
            {
                "system": "http://terminology.hl7.org/CodeSystem/condition-clinical",
                "code": "active",
                "display": "Activo"
            }
        ]
    },
    "code": {
        "coding": [
            {
                "system": "http://hl7.org/fhir/sid/icd-10", #de dónde viene el código. Esta URL le dice al sistema que el código que sigue es CIE-10. Es como decir *"lo que viene es un código del libro de la OMS"*.
                "code": "J00", #Aquí vive el CIE-10. 
                "display": "Rinofaringitis" #el nombre legible del diagnóstico. Para humanos
            }
        ]
    },
    "subject": {
        "reference": "Patient/pac-001"
    },
    "encounter": {
        "reference": "Encounter/enc-001"
    }
}

print(json.dumps(diagnostico, indent=2, ensure_ascii=False))

{
  "resourceType": "Condition",
  "id": "con-001",
  "clinicalStatus": {
    "coding": [
      {
        "system": "http://terminology.hl7.org/CodeSystem/condition-clinical",
        "code": "active",
        "display": "Activo"
      }
    ]
  },
  "code": {
    "coding": [
      {
        "system": "http://hl7.org/fhir/sid/icd-10",
        "code": "J00",
        "display": "Rinofaringitis"
      }
    ]
  },
  "subject": {
    "reference": "Patient/pac-001"
  },
  "encounter": {
    "reference": "Encounter/enc-001"
  }
}


## MedicationStatement

Después del diagnóstico, tú formulas. Escribes el medicamento, la dosis, la frecuencia, la vía.
Eso es el MedicationStatement. Es la formulación de Ana asociada a esa consulta.

In [11]:
# CELDA 4 — El Medicamento (MedicationStatement)
# Lo que le formulaste a Ana

medicamento = {
    "resourceType": "MedicationStatement",
    "id": "med-001",
    "status": "active",
    "medicationCodeableConcept": { #Es el nombre y código del medicamento.
        "coding": [
            {
                "system": "http://www.whocc.no/atc", #Aquí usamos ATC — el sistema de clasificación de medicamentos de la OMS. Es el equivalente del CIE-10 pero para medicamentos.
                "code": "J01CA04",
                "display": "Amoxicilina 500mg"
            }
        ]
    },
    "subject": {
        "reference": "Patient/pac-001"
    },
    "context": {
        "reference": "Encounter/enc-001"
    },
    "dosage": [
        {
            "text": "500mg cada 8 horas por 7 días",
            "timing": {
                "repeat": {
                    "frequency": 3,
                    "period": 1,
                    "periodUnit": "d"
                }
            },
            "route": {
                "coding": [
                    {
                        "system": "http://snomed.info/sct",
                        "code": "26643006",
                        "display": "Vía oral"
                    }
                ]
            }
        }
    ]
}

print(json.dumps(medicamento, indent=2, ensure_ascii=False))

{
  "resourceType": "MedicationStatement",
  "id": "med-001",
  "status": "active",
  "medicationCodeableConcept": {
    "coding": [
      {
        "system": "http://www.whocc.no/atc",
        "code": "J01CA04",
        "display": "Amoxicilina 500mg"
      }
    ]
  },
  "subject": {
    "reference": "Patient/pac-001"
  },
  "context": {
    "reference": "Encounter/enc-001"
  },
  "dosage": [
    {
      "text": "500mg cada 8 horas por 7 días",
      "timing": {
        "repeat": {
          "frequency": 3,
          "period": 1,
          "periodUnit": "d"
        }
      },
      "route": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "26643006",
            "display": "Vía oral"
          }
        ]
      }
    }
  ]
}
